# Modelo Compacto: 1D-CNN sobre Curvas Térmicas (Modelo 4)

Este cuaderno implementa el entrenamiento y la evaluación de la arquitectura **1D-CNN** diseñada para estimar el tiempo transcurrido utilizando curvas numéricas de enfriamiento compactas.

### Características del enfoque:
1. **Entrada Compacta:** En lugar de leer imágenes térmicas pesadas `.npy`, el modelo se alimenta directamente de las 4 variables físicas precalculadas en `metadata.csv`:
   * `hot_delta_tmean_C_p95`: Temperatura media diferencial del área más caliente.
   * `hot_area_px_p95`: Área activa de la huella en píxeles.
   * `delta_tmean_C`: Cambio medio de temperatura respecto al fondo.
   * `img_tmax_C`: Temperatura máxima registrada.
2. **Ventanas Móviles 1D:** Genera ventanas deslizantes temporales de tamaño $W=5$ pasos de tiempo dentro del mismo contacto (`sequence_id`).
3. **Velocidad Extrema:** Dado que procesa únicamente vectores numéricos de baja dimensionalidad, el entrenamiento completo toma solo un par de segundos en CPU o GPU.

In [1]:
import sys
import random
from pathlib import Path

# Añadir el directorio raíz al path de Python para permitir importaciones correctas
sys.path.append(str(Path.cwd().parent))

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader

from src.models.cnn_1d import Thermal1DCNN
from src.loaders.cnn_1d_loader import Thermal1DDataset
from src.utils import SqrtScaledMSELoss, eval_cnn_1d_metrics

### 1. Semilla Global y Reproducibilidad

In [2]:
def set_seed(seed: int = 42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

set_seed(42)

### 2. Configuración General (Hiperparámetros)

In [3]:
CONFIG = {
    "epochs": 100,
    "patience": 15,
    "min_delta": 1.0,
    "batch_size": 16,
    "lr": 0.001,
    "weight_decay": 0.0005,
    "min_time_s": 0.0,
    "seq_len": 5,              # Ventana móvil de 5 mediciones
    "device": "cuda" if torch.cuda.is_available() else "cpu",
    "time_scale": 30.0,
}

### 3. Partición de Secuencias y Carga de Datasets (Sin Data Leakage)

Dividimos los datos por secuencias físicas. Para prevenir fugas de información, calculamos la media y desviación estándar de las variables físicas en el conjunto de **entrenamiento** y las aplicamos en la normalización de **validación**.

In [4]:
metadata_path = "../processed_data/metadata.csv"
df = pd.read_csv(metadata_path)

# Extraer secuencias únicas
unique_seqs = df["sequence_id"].dropna().unique()
np.random.default_rng(42).shuffle(unique_seqs)

split_idx = int(len(unique_seqs) * 0.8)
train_seq_ids = unique_seqs[:split_idx]
val_seq_ids = unique_seqs[split_idx:]

# Inicializar dataset de entrenamiento
train_ds = Thermal1DDataset(
    metadata_csv=metadata_path,
    is_train=True,
    min_time_s=CONFIG["min_time_s"],
    seq_len=CONFIG["seq_len"],
    sequence_ids=train_seq_ids
)

# Inicializar dataset de validación heredando la media y desviación del entrenamiento
val_ds = Thermal1DDataset(
    metadata_csv=metadata_path,
    is_train=False,
    min_time_s=CONFIG["min_time_s"],
    seq_len=CONFIG["seq_len"],
    sequence_ids=val_seq_ids,
    means=train_ds.means,
    stds=train_ds.stds
)

train_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"], shuffle=True)
val_loader = DataLoader(val_ds, batch_size=CONFIG["batch_size"])
train_eval_loader = DataLoader(train_ds, batch_size=CONFIG["batch_size"])

print(f"Medias del entrenamiento: {train_ds.means}")
print(f"Desviaciones del entrenamiento: {train_ds.stds}")
print(f"Muestras 1D: {len(train_ds)} train / {len(val_ds)} val")

Medias del entrenamiento: [1.4469664e+00 6.5598379e+03 6.1404103e-01 2.5443520e+01]
Desviaciones del entrenamiento: [9.6908355e-01 7.4760950e+02 7.3795575e-01 1.4781710e+00]
Muestras 1D: 1015 train / 301 val


C:\Users\esteb\AppData\Local\Temp\ipykernel_7644\1833258232.py:6: UserWarning: you are shuffling a 'StringArray' object which is not a subclass of 'Sequence'; `shuffle` is not guaranteed to behave correctly. E.g., non-numpy array/tensor objects with view semantics may contain duplicates after shuffling.
  np.random.default_rng(42).shuffle(unique_seqs)


### 4. Inicialización del Modelo, Pérdida y Optimización

In [5]:
dev = CONFIG["device"]
# El modelo recibe 4 canales (uno por cada variable física)
model = Thermal1DCNN(in_channels=4).to(dev)

crit = SqrtScaledMSELoss(scale=CONFIG["time_scale"])
opt = torch.optim.AdamW(model.parameters(), lr=CONFIG["lr"], weight_decay=CONFIG["weight_decay"])
scheduler = torch.optim.lr_scheduler.MultiStepLR(opt, milestones=[30, 60, 80], gamma=0.8)

print(f"Modelo 1D-CNN cargado con {sum(p.numel() for p in model.parameters()):,} parámetros")

Modelo 1D-CNN cargado con 9,057 parámetros


### 5. Ciclo de Entrenamiento e Impresión de Métricas Unificadas

In [6]:
best_mae = float("inf")
no_imp = 0

for ep in range(1, CONFIG["epochs"] + 1):
    model.train()
    train_loss, n = 0.0, 0
    for x_seq, y in train_loader:
        x_seq, y = x_seq.to(dev), y.to(dev)
        opt.zero_grad(set_to_none=True)
        loss = crit(model(x_seq), y)
        loss.backward()
        opt.step()
        train_loss += loss.item() * x_seq.size(0)
        n += x_seq.size(0)

    scheduler.step()

    # Evaluar métricas unificadas en segundos reales
    val_m = eval_cnn_1d_metrics(model, val_loader, dev)
    train_m = eval_cnn_1d_metrics(model, train_eval_loader, dev)

    v_mae = val_m["mae"]
    is_best = v_mae < best_mae - CONFIG["min_delta"]
    if is_best: 
        best_mae, no_imp = v_mae, 0
        torch.save(model.state_dict(), "../CNN1D_best.pt")
    else: 
        no_imp += 1
    
    print(f"Ep {ep:03d} | Loss: {train_loss/n:.4f} | "
          f"TrMAE: {train_m['mae']:5.2f}s | ValMAE: {v_mae:5.2f}s | "
          f"RMSE: {val_m['rmse']:5.2f}s | R2: {val_m['r2']:.4f} | MAPE: {val_m['mape']:5.2f}% | "
          f"Acc60: {val_m['acc60']:.2f}% | Acc120: {val_m['acc120']:.2f}% {'*' if is_best else ''}")
    
    if no_imp >= CONFIG["patience"]:
        print(f"Early stop alcanzado. Mejor Val MAE: {best_mae:.2f}s")
        break

Ep 001 | Loss: 6.0381 | TrMAE: 211.00s | ValMAE: 216.32s | RMSE: 269.18s | R2: -1.8009 | MAPE: 96.38% | Acc60: 20.60% | Acc120: 34.88% *
Ep 002 | Loss: 4.4888 | TrMAE: 195.87s | ValMAE: 201.76s | RMSE: 255.86s | R2: -1.5305 | MAPE: 86.19% | Acc60: 22.92% | Acc120: 38.54% *
Ep 003 | Loss: 2.8004 | TrMAE: 160.90s | ValMAE: 166.26s | RMSE: 223.91s | R2: -0.9380 | MAPE: 63.10% | Acc60: 31.89% | Acc120: 47.51% *
Ep 004 | Loss: 1.5755 | TrMAE: 124.64s | ValMAE: 133.83s | RMSE: 191.46s | R2: -0.4169 | MAPE: 52.09% | Acc60: 41.53% | Acc120: 60.13% *
Ep 005 | Loss: 0.9005 | TrMAE: 95.56s | ValMAE: 108.42s | RMSE: 159.47s | R2: 0.0171 | MAPE: 52.62% | Acc60: 50.17% | Acc120: 73.09% *
Ep 006 | Loss: 0.6537 | TrMAE: 93.36s | ValMAE: 103.34s | RMSE: 149.99s | R2: 0.1304 | MAPE: 52.68% | Acc60: 46.51% | Acc120: 72.43% *
Ep 007 | Loss: 0.5705 | TrMAE: 88.00s | ValMAE: 109.00s | RMSE: 152.66s | R2: 0.0991 | MAPE: 64.26% | Acc60: 44.85% | Acc120: 69.10% 
Ep 008 | Loss: 0.5467 | TrMAE: 86.04s | ValMAE: 